# Notebook LIGHT — YouTube AI Recommendations + Qwen QLoRA

Este notebook documenta y ejecuta la versión ligera que corre en Gradio. Mantiene el flujo de la app original `youtube-ai-recomendations`, con el adaptador Qwen QLoRA cargado localmente y el gate XGBoost integrado.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
print(ROOT)
print((ROOT/'app.py').exists())
print((ROOT/'models/qwen_marketing_qlora/adapter_config.json').exists())

Repo root detectado.
app.py: True
QLoRA adapter_config: True


## Componentes incluidos

- Gradio app: `app.py`
- Clasificador principal: `models/best_model.joblib`
- QLoRA/Qwen: `models/qwen_marketing_qlora/`
- Segundo modelo XGBoost: `models/xgboost_paid_ads.joblib`
- Módulos multimodales: OCR, transcripción, visual composition, sentimiento, policy evaluator, LLM provider.

In [1]:
import json, pandas as pd
print(pd.read_csv('outputs/model_comparison.csv').to_string(index=False))

 accuracy  precision   recall       f1  roc_auc   tn   fp   fn   tp                  model
 0.687509   0.482324 0.567364 0.521399 0.732783 7090 2504 1779 2333    logistic_regression
 0.746097   0.627625 0.377918 0.471767 0.744242 8672  922 2558 1554          random_forest
 0.755509   0.722125 0.300827 0.424721 0.735366 9118  476 2875 1237             linear_svc
 0.760397   0.825984 0.255107 0.389818 0.755184 9373  221 3063 1049 hist_gradient_boosting


In [1]:
import json
meta=json.load(open('models/xgboost_paid_ads_metadata.json'))
print(json.dumps(meta['metrics'], indent=2, ensure_ascii=False))

{
  "paid_performance_score": {
    "mae": 8.812764919048815,
    "rmse": 10.902629646787105,
    "r2": 0.37750773469891763,
    "mape_pct": 19.457599943242517,
    "test_mean": 50.607021121988566
  },
  "cpm": {
    "mae": 4.50505755053344,
    "rmse": 5.910916344158139,
    "r2": 0.5086430751881359,
    "mape_pct": 149.55851720913512,
    "test_mean": 10.011057839993084
  }
}


## Ejecutar Gradio

```bash
python app.py
```

El flujo predictivo queda así: regresión logística → si probabilidad ≥ 51% → XGBoost estima rendimiento pagado, CPM, impresiones por dólar y nicho.

In [ ]:
# Smoke test opcional desde la raíz del repo
# !python tests/smoke_test.py